# MVA · NB05A3 — F0 implementation freeze
**v1.0.2 · corrected ClinGen control logic · U2OS F0V + hTERT-RPE1 F0M**

This notebook freezes the execution architecture. It does not generate biological results and does not assign PS3/BS3.

In [ ]:
import os
import json
from pathlib import Path
import pandas as pd
try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    ROOT = Path(os.environ.get("MVA_ROOT", "/path/to/MVA_Hackathon_2026"))
except Exception:
    ROOT=Path('.')
PUBLIC=ROOT/'03_PUBLIC_ARTIFACTS'
PRIVATE=ROOT/'02_RESULTS_PRIVATE'
VERSION='v1_0_2'
SCHEMA='MVA-NB05A3-F0-SPLIT-ASSAY-1.0.2'
print(ROOT, SCHEMA)

## 1. Upstream design and curation locks

In [ ]:
a=json.loads((PUBLIC/'NB05A_HANDOFF_v1_0_0.json').read_text())
c=json.loads((PUBLIC/'NB05A2_MANUAL_CURATION_HANDOFF_v1_0_2.json').read_text())
assert a['status']=='DESIGN_COMPLETE_AWAITING_EXPERIMENT'
assert c['status']=='CONTROL_PANEL_CURATION_NUMERIC_GATE_MET_AWAITING_ASSAY_VALIDATION'
assert c['confirmed_independent_controls']['total']==11
print('Upstream locks: PASS')

## 2. ClinGen correction

v1.0.1 was stricter than the ClinGen SVI framework. The corrected project rule requires 11 total independent mixed controls, but allows the ClinGen-modeled possibility of one indeterminate control readout. Multiple indeterminate/discordant controls do not get repaired by post-hoc threshold tuning; they trigger downgrade or formal OddsPath analysis.

In [ ]:
qc=pd.read_csv(PUBLIC/'NB05A3_F0_SPLIT_QC_GATES_v1_0_2.csv')
display(qc)

## 3. F0V implementation

**U2OS BUBR1 depletion/complementation**, anchored to PMID 20516114. N1002K is held out. The 11 control variants define assay validation; cDNA truncating controls are interpreted only at the protein-function level.

In [ ]:
f0v=pd.read_csv(PUBLIC/'NB05A3_F0V_VARIANT_ASSAY_MANIFEST_v1_0_2.csv')
assert f0v['counts_for_clingen_validation'].astype(str).str.lower().eq('true').sum()==11
assert f0v['held_out_test'].astype(str).str.lower().eq('true').sum()==1
display(f0v)

## 4. F0M implementation

**hTERT-RPE1 endogenous isogenic allelic series**. This layer resolves abundance-limited vs function-limited biology and causal rescue; it is not used to inflate ClinGen validation-control counts.

In [ ]:
f0m=pd.read_csv(PUBLIC/'NB05A3_F0M_MECHANISM_MANIFEST_v1_0_2.csv')
assert len(f0m)==5
display(f0m)

## 5. Implementation freeze

In [ ]:
impl=pd.read_csv(PUBLIC/'NB05A3_LAB_IMPLEMENTATION_FREEZE_v1_0_2.csv')
assert (impl['status']=='FROZEN').all()
display(impl)

## 6. Stop condition

NB05B remains blocked until real F0V/F0M observations exist. No drug selection. No automatic PS3/BS3 assignment.